In [80]:
import ollama
import json
import re
from collections import deque

## SLM Models

In [81]:
MODEL_QWEN_2B = "qwen3.5:2b"

## Load Data

In [82]:
with open("data/grids_5.json", "r") as f:
    grids_5 = json.load(f)

with open("data/grids_7.json", "r") as f:
    grids_7 = json.load(f)

with open("data/grids_10.json", "r") as f:
    grids_10 = json.load(f)

## Helper Functions

### Query SLM

In [83]:
def ask_slm(messages, model):
    # print("Asking SLM...")
    response = ollama.chat(
        model=model,
        messages=messages,
        think=False,
        options={"temperature": 0.0, "seed": 42}
    )

    return response["message"]["content"]

### Generate SLM System Prompt

In [84]:
def generate_system_prompt(grid_size=5):
    SYSTEM_PROMPT = f"""
    You are a robot path planner operating in a {grid_size}×{grid_size} grid world.

    Environment
        - Coordinates are represented as [row, column].
        - Rows and columns are indexed from 0 to {grid_size - 1}.
        - Row indices increase from top to bottom.
        - Column indices increase from left to right.

    Movement rules:
        - NORTH: [row - 1, column]
        - SOUTH: [row + 1, column]
        - EAST:  [row, column + 1]
        - WEST:  [row, column - 1]
        
        Each action moves the robot exactly one cell. Only the specified coordinate changes for each action.

    Constraints
        - Never enter an obstacle cell.
        - Never leave the grid boundaries.
        - Each action moves exactly one cell.
        - Return the possible next action.
        - Any valid action is acceptable.

    Planning procedure:
        1. Start at the given starting position.
        2. Select an action that moves toward the goal while avoiding obstacles.
        3. Never choose an action that leaves the grid or enters an obstacle.
        4. A feedback will be provided after each action to indicate whether the action was valid or invalid.
        5. If the action was valid, plan the next action based on the robot position.
        6. If the action was invalid, plan a different action.

    Output format
        - Return only a valid JSON object.
        - The move direction must be one of the following: "NORTH", "SOUTH", "EAST", or "WEST".
        - Example response: {{"action": "SOUTH"}}
        - Do not include explanations, Markdown fences, intermediate positions, or additional text.
    """

    return SYSTEM_PROMPT.strip()

### Validate Robot Moves

In [85]:
MOVES = {
    "NORTH": (-1, 0),
    "SOUTH": (1, 0),
    "EAST": (0, 1),
    "WEST": (0, -1),
}

In [86]:
def validate_move(position, action, obstacles, grid_size=5):
    action = action.upper()

    if action not in MOVES:
        return {
            "valid": False,
            "reason": "Unknown action"
        }

    row, col = position
    dr, dc = MOVES[action]
    next_position = (row + dr, col + dc)

    if not (
        0 <= next_position[0] < grid_size
        and 0 <= next_position[1] < grid_size
    ):
        return {
            "valid": False,
            "reason": "Move leaves grid boundaries"
        }

    if next_position in {tuple(p) for p in obstacles}:
        return {
            "valid": False,
            "reason": "Move enters an obstacle"
        }

    return {
        "valid": True,
        "next_position": list(next_position)
    }

### Validate SLM JSON Response

In [87]:
def validate_json(json_str):
    json_str = json_str.strip()
    json_str = re.sub(r"^```(?:json)?\s*", "", json_str)
    json_str = re.sub(r"\s*```$", "", json_str)
    try:
        data = json.loads(json_str)
        if isinstance(data, dict) and "action" in data:
            return True, data['action']
        else:
            return False, "INVALID_JSON"
    except json.JSONDecodeError:
        return False, "INVALID_JSON"

### Find Neighboring Obstacles

In [88]:
def get_neighboring_obstacles(position, obstacles):
    row, col = position
    neighboring_obstacles = []

    for dr, dc in MOVES.values():
        neighbor = (row + dr, col + dc)
        if neighbor in {tuple(p) for p in obstacles}:
            neighboring_obstacles.append(list(neighbor))

    return neighboring_obstacles

### Get The Shortest Path

In [89]:
def get_shortest_path(start, goal, obstacles, grid_size=5):
    queue = deque([(start, [])])
    visited = set()
    visited.add(tuple(start))

    while queue:
        current_position, path = queue.popleft()

        if current_position == goal:
            return path

        for action, (dr, dc) in MOVES.items():
            next_position = [current_position[0] + dr, current_position[1] + dc]

            if (
                0 <= next_position[0] < grid_size
                and 0 <= next_position[1] < grid_size
                and tuple(next_position) not in visited
                and next_position not in obstacles
            ):
                visited.add(tuple(next_position))
                queue.append((next_position, path + [action]))

    return None

### Generate Solution Summary

In [90]:
def create_solution_summary(solution_details):
    total_grids = len(solution_details)
    solved_grids = 0
    solved_path_differences = 0
    solved_path_length = 0
    for detail in solution_details:
        if detail["is_solved"]:
            solved_grids += 1
            solved_path_differences += detail["path_difference"]
            solved_path_length += len(detail["slm_moves"])
    unsolved_grids = total_grids - solved_grids

    return {
        "total_grids": total_grids,
        "solved_grids": solved_grids,
        "unsolved_grids": unsolved_grids,
        "average_solved_path_difference": solved_path_differences / solved_grids if solved_grids > 0 else 0,
        "average_solved_path_length": solved_path_length  / solved_grids if solved_grids > 0 else 0
    }


## Grid Solver

In [91]:
def solve_grids(grids, model, is_local_context, is_self_correcting):
    grid_size = len(grids[0]["grid"]) if grids else 0
    solved_grids = 0
    solution_details = []

    for grid_info in grids:
        grid = grid_info["grid"]

        start_position = grid_info["start"]
        current_position = start_position
        goal_position = grid_info["goal"]
        obstacles = grid_info["obstacles"]
        moves_sequence = []
        total_moves = 0
        attempted_moves = 0
        path_difference = 0
        is_solved = False
        has_invalid_move = False

        

        print(f"Grid:\n{'\n'.join(' '.join(row) for row in grid)}")

        initial_prompt = f"Starting position: {start_position}\nGoal position: {goal_position}\nObstacles: {obstacles}"
        

        system_prompt = generate_system_prompt(grid_size=len(grid))

        messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": initial_prompt}
            ]


        while attempted_moves < (grid_size * grid_size):
            print(f"Attempted moves: {attempted_moves}, Total executed moves: {total_moves}, Current position: {current_position}")
            attempted_moves += 1
            system_message = messages[0]

            if is_local_context:
                recent_messages = messages[1:][-4:]
                                
                context = [
                    system_message,
                    *recent_messages
                ]
            else:
                initial_user_message = messages[1]
                recent_messages = messages[2:][-4:]

                context = [
                    system_message,
                    initial_user_message,
                    *recent_messages
                ]
                

            response = ask_slm(context, model)
            print(f"SLM response: {response}")
            
            is_valid_json, action = validate_json(response)

            if not is_valid_json:
                print(f"Invalid JSON response: {response}. Retrying...")


                messages.append({
                    "role": "user",
                    "content": (
                        f"The action was invalid.\n"
                        f"Current position: {current_position}\n"
                        "Please provide a valid JSON object with an action."
                    ),
                })

                continue

            validation_result = validate_move(current_position, action, obstacles)

            if validation_result["valid"]:
                total_moves += 1
                moves_sequence.append(action)
                # print(f"Valid move: {action}.")
                current_position = validation_result['next_position']
                is_solved = current_position == goal_position

                if is_solved:
                    solved_grids += 1
                    break

                if is_local_context:
                    neighboring_obstacles = get_neighboring_obstacles(current_position, obstacles)
                    
                    messages.append({
                        "role": "user",
                        "content": (
                            f"The action was valid.\n"
                            f"Current position: {current_position}\n"
                            f"Goal position: {goal_position}\n"
                            f"Neighboring Obstacles: {neighboring_obstacles}\n"
                        ),
                    })
                else:
                    messages.append({
                        "role": "user",
                        "content": (
                            f"The action was valid.\n"
                            f"Current position: {current_position}\n"
                            f"Goal position: {goal_position}\n"
                            f"Obstacles: {obstacles}\n"
                        ),
                    })

                
            else:
                has_invalid_move = True
                messages.append({
                    "role": "user",
                    "content": f"Invalid move: {action}.\nCurrent position: {current_position}.\n Reason: {validation_result['reason']}. Retrying..."
                })

                if not is_self_correcting:
                    print(f"Invalid move: {action}. Reason: {validation_result['reason']}.")
                    break

        # print(f"Response:\n{response}\n")
        print("SLM Path Planning Results:")

        if is_solved:
            print("Goal reached!")
        elif has_invalid_move and not is_self_correcting:
            print("Encountered an invalid move. Stopping further attempts.")
        else:
            print("Failed to reach the goal within 25 attempts.")

        print(f"Total attempted moves: {attempted_moves}")
        print(f"Total executed moves: {total_moves}")
        # print(f"Moves sequence: {moves_sequence}")

        print("Shortest Path Planning Results:")
        shortest_path = get_shortest_path(start_position, goal_position, obstacles)
        if shortest_path is not None:
            print(f"Shortest path to goal: {shortest_path}")
        else:
            print("Failed to find a path to the goal.")

        if shortest_path is not None and is_solved:
            print("Different between SLM generated path and the shortest path:")
            path_difference = len(moves_sequence) - len(shortest_path)
            print(f"SLM took {path_difference} more moves than the shortest path.")
        
        solution_details.append({
            "start": start_position,
            "goal": goal_position,
            "obstacles": obstacles,
            "slm_moves": moves_sequence,
            "bfs_moves": shortest_path,
            "is_solved": is_solved,
            "has_invalid_move": has_invalid_move,
            "path_difference": path_difference
        })

    print(f"Total solved grids: {solved_grids} out of {len(grids)}")

    return solution_details

## Global Context with Self Correction

### 5x5 Grids

In [92]:
solution_details_global_self_correction_grid_5 = solve_grids(
    grids=grids_5, 
    model=MODEL_QWEN_2B, 
    is_local_context=False, 
    is_self_correcting=True
)

Grid:
S . . . B
B . . . .
. G . . .
. . . . B
. . . B .
Attempted moves: 0, Total executed moves: 0, Current position: [0, 0]
SLM response: {"action": "SOUTH"}
Attempted moves: 1, Total executed moves: 0, Current position: [0, 0]
SLM response: {"action": "EAST"}
Attempted moves: 2, Total executed moves: 1, Current position: [0, 1]
SLM response: {"action": "SOUTH"}
Attempted moves: 3, Total executed moves: 2, Current position: [1, 1]
SLM response: {"action": "SOUTH"}
SLM Path Planning Results:
Goal reached!
Total attempted moves: 4
Total executed moves: 3
Shortest Path Planning Results:
Shortest path to goal: ['EAST', 'SOUTH', 'SOUTH']
Different between SLM generated path and the shortest path:
SLM took 0 more moves than the shortest path.
Grid:
. . B . S
. . B B .
. . . . .
G . . B .
. . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [0, 4]
SLM response: {"action": "SOUTH"}
Attempted moves: 1, Total executed moves: 1, Current position: [1, 4]
SLM response: {"actio

In [93]:
create_solution_summary(solution_details_global_self_correction_grid_5)

{'total_grids': 100,
 'solved_grids': 87,
 'unsolved_grids': 13,
 'average_solved_path_difference': 0.8735632183908046,
 'average_solved_path_length': 4.126436781609195}

### 7x7 Grid

In [94]:
solution_details_global_self_correction_grid_7 = solve_grids(
    grids=grids_7, 
    model=MODEL_QWEN_2B, 
    is_local_context=False, 
    is_self_correcting=True
)

Grid:
. . . . . . .
. . . . . . .
. . . . . . B
B B . B G . .
. . B . . . .
B B . . . . .
. S . . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "NORTH"}
Attempted moves: 1, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "SOUTH"}
Attempted moves: 2, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "EAST"}
Attempted moves: 3, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "WEST"}
Attempted moves: 4, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "NORTH"}
Attempted moves: 5, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "NORTH"}
Attempted moves: 6, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "SOUTH"}
Attempted moves: 7, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "EAST"}
Attempted moves: 8, Total executed moves: 0, Current position: [6, 

In [95]:
create_solution_summary(solution_details_global_self_correction_grid_7) 

{'total_grids': 100,
 'solved_grids': 30,
 'unsolved_grids': 70,
 'average_solved_path_difference': 0.3333333333333333,
 'average_solved_path_length': 3.7333333333333334}

### 10x10 Grid

In [96]:
solution_details_global_self_correction_grid_10 = solve_grids(
    grids=grids_10, 
    model=MODEL_QWEN_2B, 
    is_local_context=False, 
    is_self_correcting=True
)

Grid:
. . . . . . . G B .
B . . . . . . . . .
. . . . . . . . . .
. . B . . . . . B B
. . . . . . . . . .
. . B . . . . . . B
. . B . . . . . . S
. B . . . . . . . .
. . . . B . . . . B
B . . . . . . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "NORTH"}
Attempted moves: 1, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "WEST"}
Attempted moves: 2, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "SOUTH"}
Attempted moves: 3, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "EAST"}
Attempted moves: 4, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "NORTH"}
Attempted moves: 5, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "NORTH"}
Attempted moves: 6, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "SOUTH"}
Attempted moves: 7, Total executed moves: 0, Current position: [6, 9

In [97]:
create_solution_summary(solution_details_global_self_correction_grid_10)

{'total_grids': 100,
 'solved_grids': 6,
 'unsolved_grids': 94,
 'average_solved_path_difference': 0.6666666666666666,
 'average_solved_path_length': 3.8333333333333335}

## Global Context without Self Correction

### 5x5 Grid

In [98]:
solution_details_global_without_self_correction_grid_5 = solve_grids(
    grids=grids_5, 
    model=MODEL_QWEN_2B, 
    is_local_context=False, 
    is_self_correcting=False
)

Grid:
S . . . B
B . . . .
. G . . .
. . . . B
. . . B .
Attempted moves: 0, Total executed moves: 0, Current position: [0, 0]
SLM response: {"action": "SOUTH"}
Invalid move: SOUTH. Reason: Move enters an obstacle.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 1
Total executed moves: 0
Shortest Path Planning Results:
Shortest path to goal: ['EAST', 'SOUTH', 'SOUTH']
Grid:
. . B . S
. . B B .
. . . . .
G . . B .
. . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [0, 4]
SLM response: {"action": "SOUTH"}
Attempted moves: 1, Total executed moves: 1, Current position: [1, 4]
SLM response: {"action": "WEST"}
Invalid move: WEST. Reason: Move enters an obstacle.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 2
Total executed moves: 1
Shortest Path Planning Results:
Shortest path to goal: ['SOUTH', 'SOUTH', 'WEST', 'WEST', 'SOUTH', 'WEST', 'WEST']
Grid:
. .

In [99]:
create_solution_summary(solution_details_global_without_self_correction_grid_5)

{'total_grids': 100,
 'solved_grids': 55,
 'unsolved_grids': 45,
 'average_solved_path_difference': 0.03636363636363636,
 'average_solved_path_length': 2.8}

### 7x7 Grid

In [100]:
solution_details_global_without_self_correction_grid_7 = solve_grids(
    grids=grids_7, 
    model=MODEL_QWEN_2B, 
    is_local_context=False, 
    is_self_correcting=False
)

Grid:
. . . . . . .
. . . . . . .
. . . . . . B
B B . B G . .
. . B . . . .
B B . . . . .
. S . . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "NORTH"}
Invalid move: NORTH. Reason: Move leaves grid boundaries.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 1
Total executed moves: 0
Shortest Path Planning Results:
Failed to find a path to the goal.
Grid:
. . B . . . .
. . . . . . .
B . . . . . B
. . . B . B .
. . B . . . .
. G . . . . B
. S . . . . B
Attempted moves: 0, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "NORTH"}
Invalid move: NORTH. Reason: Move leaves grid boundaries.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 1
Total executed moves: 0
Shortest Path Planning Results:
Failed to find a path to the goal.
Grid:
. . B . . . .
. B . . . . .
. . . . . . .
B . . . . . .
. . . B 

In [101]:
create_solution_summary(solution_details_global_without_self_correction_grid_7)

{'total_grids': 100,
 'solved_grids': 25,
 'unsolved_grids': 75,
 'average_solved_path_difference': 0.0,
 'average_solved_path_length': 3.2}

### 10x10 Grid

In [102]:
solution_details_global_without_self_correction_grid_10 = solve_grids(
    grids=grids_10, 
    model=MODEL_QWEN_2B, 
    is_local_context=False, 
    is_self_correcting=False
)

Grid:
. . . . . . . G B .
B . . . . . . . . .
. . . . . . . . . .
. . B . . . . . B B
. . . . . . . . . .
. . B . . . . . . B
. . B . . . . . . S
. B . . . . . . . .
. . . . B . . . . B
B . . . . . . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "NORTH"}
Invalid move: NORTH. Reason: Move leaves grid boundaries.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 1
Total executed moves: 0
Shortest Path Planning Results:
Failed to find a path to the goal.
Grid:
. . . . . . . . . .
. . . . . . B . B .
. . . . . . B S . .
. . . . . . . . . .
. B . . B B . . . .
. . B . . . . . . .
. . . . . . . G . .
. . . . . . . . . B
. . . . . . . . . .
. . . B . B . . . B
Attempted moves: 0, Total executed moves: 0, Current position: [2, 7]
SLM response: {"action": "SOUTH"}
Invalid move: SOUTH. Reason: Move leaves grid boundaries.
SLM Path Planning Results:
Encountered an invalid move. Stopping 

In [103]:
create_solution_summary(solution_details_global_without_self_correction_grid_10)

{'total_grids': 100,
 'solved_grids': 5,
 'unsolved_grids': 95,
 'average_solved_path_difference': 0.0,
 'average_solved_path_length': 3.0}

## Local Context with Self Correction

### 5x5 Grid

In [104]:
solution_details_local_self_correction_grid_5 = solve_grids(
    grids=grids_5, 
    model=MODEL_QWEN_2B, 
    is_local_context=True, 
    is_self_correcting=True
)

Grid:
S . . . B
B . . . .
. G . . .
. . . . B
. . . B .
Attempted moves: 0, Total executed moves: 0, Current position: [0, 0]
SLM response: {"action": "SOUTH"}
Attempted moves: 1, Total executed moves: 0, Current position: [0, 0]
SLM response: {"action": "EAST"}
Attempted moves: 2, Total executed moves: 1, Current position: [0, 1]
SLM response: {"action": "SOUTH"}
Attempted moves: 3, Total executed moves: 2, Current position: [1, 1]
SLM response: {"action": "SOUTH"}
SLM Path Planning Results:
Goal reached!
Total attempted moves: 4
Total executed moves: 3
Shortest Path Planning Results:
Shortest path to goal: ['EAST', 'SOUTH', 'SOUTH']
Different between SLM generated path and the shortest path:
SLM took 0 more moves than the shortest path.
Grid:
. . B . S
. . B B .
. . . . .
G . . B .
. . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [0, 4]
SLM response: {"action": "SOUTH"}
Attempted moves: 1, Total executed moves: 1, Current position: [1, 4]
SLM response: {"actio

In [105]:
create_solution_summary(solution_details_local_self_correction_grid_5)

{'total_grids': 100,
 'solved_grids': 80,
 'unsolved_grids': 20,
 'average_solved_path_difference': 0.425,
 'average_solved_path_length': 3.525}

### 7x7 Grid

In [106]:
solution_details_local_self_correction_grid_7 = solve_grids(
    grids=grids_7, 
    model=MODEL_QWEN_2B, 
    is_local_context=True, 
    is_self_correcting=True
)

Grid:
. . . . . . .
. . . . . . .
. . . . . . B
B B . B G . .
. . B . . . .
B B . . . . .
. S . . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "NORTH"}
Attempted moves: 1, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "SOUTH"}
Attempted moves: 2, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "EAST"}
Attempted moves: 3, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "WEST"}
Attempted moves: 4, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "NORTH"}
Attempted moves: 5, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "SOUTH"}
Attempted moves: 6, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "EAST"}
Attempted moves: 7, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "WEST"}
Attempted moves: 8, Total executed moves: 0, Current position: [6, 1

In [107]:
create_solution_summary(solution_details_global_self_correction_grid_7)

{'total_grids': 100,
 'solved_grids': 30,
 'unsolved_grids': 70,
 'average_solved_path_difference': 0.3333333333333333,
 'average_solved_path_length': 3.7333333333333334}

### 10x10 Grid

In [109]:
solution_details_local_self_correction_grid_10 = solve_grids(
    grids=grids_10, 
    model=MODEL_QWEN_2B, 
    is_local_context=True, 
    is_self_correcting=True
)

Grid:
. . . . . . . G B .
B . . . . . . . . .
. . . . . . . . . .
. . B . . . . . B B
. . . . . . . . . .
. . B . . . . . . B
. . B . . . . . . S
. B . . . . . . . .
. . . . B . . . . B
B . . . . . . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "NORTH"}
Attempted moves: 1, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "WEST"}
Attempted moves: 2, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "SOUTH"}
Attempted moves: 3, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "EAST"}
Attempted moves: 4, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "NORTH"}
Attempted moves: 5, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "WEST"}
Attempted moves: 6, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "NORTH"}
Attempted moves: 7, Total executed moves: 0, Current position: [6, 9]

In [110]:
create_solution_summary(solution_details_global_self_correction_grid_10)

{'total_grids': 100,
 'solved_grids': 6,
 'unsolved_grids': 94,
 'average_solved_path_difference': 0.6666666666666666,
 'average_solved_path_length': 3.8333333333333335}

## Local Context without Self Correction

### 5x5 Grid

In [111]:
solution_details_local_without_self_correction_grid_5 = solve_grids(
    grids=grids_5, 
    model=MODEL_QWEN_2B, 
    is_local_context=True, 
    is_self_correcting=False
)

Grid:
S . . . B
B . . . .
. G . . .
. . . . B
. . . B .
Attempted moves: 0, Total executed moves: 0, Current position: [0, 0]
SLM response: {"action": "SOUTH"}
Invalid move: SOUTH. Reason: Move enters an obstacle.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 1
Total executed moves: 0
Shortest Path Planning Results:
Shortest path to goal: ['EAST', 'SOUTH', 'SOUTH']
Grid:
. . B . S
. . B B .
. . . . .
G . . B .
. . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [0, 4]
SLM response: {"action": "SOUTH"}
Attempted moves: 1, Total executed moves: 1, Current position: [1, 4]
SLM response: {"action": "WEST"}
Invalid move: WEST. Reason: Move enters an obstacle.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 2
Total executed moves: 1
Shortest Path Planning Results:
Shortest path to goal: ['SOUTH', 'SOUTH', 'WEST', 'WEST', 'SOUTH', 'WEST', 'WEST']
Grid:
. .

In [112]:
create_solution_summary(solution_details_local_without_self_correction_grid_5)

{'total_grids': 100,
 'solved_grids': 57,
 'unsolved_grids': 43,
 'average_solved_path_difference': 0.03508771929824561,
 'average_solved_path_length': 2.807017543859649}

### 7x7 Grid

In [113]:
solution_details_local_without_self_correction_grid_7 = solve_grids(
    grids=grids_7, 
    model=MODEL_QWEN_2B, 
    is_local_context=True, 
    is_self_correcting=False
)

Grid:
. . . . . . .
. . . . . . .
. . . . . . B
B B . B G . .
. . B . . . .
B B . . . . .
. S . . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "NORTH"}
Invalid move: NORTH. Reason: Move leaves grid boundaries.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 1
Total executed moves: 0
Shortest Path Planning Results:
Failed to find a path to the goal.
Grid:
. . B . . . .
. . . . . . .
B . . . . . B
. . . B . B .
. . B . . . .
. G . . . . B
. S . . . . B
Attempted moves: 0, Total executed moves: 0, Current position: [6, 1]
SLM response: {"action": "NORTH"}
Invalid move: NORTH. Reason: Move leaves grid boundaries.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 1
Total executed moves: 0
Shortest Path Planning Results:
Failed to find a path to the goal.
Grid:
. . B . . . .
. B . . . . .
. . . . . . .
B . . . . . .
. . . B 

In [114]:
create_solution_summary(solution_details_local_without_self_correction_grid_7)

{'total_grids': 100,
 'solved_grids': 24,
 'unsolved_grids': 76,
 'average_solved_path_difference': 0.0,
 'average_solved_path_length': 3.2083333333333335}

### 10x10 Grid

In [115]:
solution_details_local_without_self_correction_grid_10 = solve_grids(
    grids=grids_10, 
    model=MODEL_QWEN_2B, 
    is_local_context=True, 
    is_self_correcting=False
)

Grid:
. . . . . . . G B .
B . . . . . . . . .
. . . . . . . . . .
. . B . . . . . B B
. . . . . . . . . .
. . B . . . . . . B
. . B . . . . . . S
. B . . . . . . . .
. . . . B . . . . B
B . . . . . . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [6, 9]
SLM response: {"action": "NORTH"}
Invalid move: NORTH. Reason: Move leaves grid boundaries.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 1
Total executed moves: 0
Shortest Path Planning Results:
Failed to find a path to the goal.
Grid:
. . . . . . . . . .
. . . . . . B . B .
. . . . . . B S . .
. . . . . . . . . .
. B . . B B . . . .
. . B . . . . . . .
. . . . . . . G . .
. . . . . . . . . B
. . . . . . . . . .
. . . B . B . . . B
Attempted moves: 0, Total executed moves: 0, Current position: [2, 7]
SLM response: {"action": "SOUTH"}
Invalid move: SOUTH. Reason: Move leaves grid boundaries.
SLM Path Planning Results:
Encountered an invalid move. Stopping 

In [116]:
create_solution_summary(solution_details_local_without_self_correction_grid_10)

{'total_grids': 100,
 'solved_grids': 5,
 'unsolved_grids': 95,
 'average_solved_path_difference': 0.0,
 'average_solved_path_length': 3.0}

In [117]:
with open("solutions/solution_details_global_self_correction_grid_5.json", "w") as f:
    json.dump(solution_details_global_self_correction_grid_5, f, indent=4)
with open("solutions/solution_details_global_self_correction_grid_7.json", "w") as f:
    json.dump(solution_details_global_self_correction_grid_7, f, indent=4)
with open("solutions/solution_details_global_self_correction_grid_10.json", "w") as f:
    json.dump(solution_details_global_self_correction_grid_10, f, indent=4)
with open("solutions/solution_details_global_without_self_correction_grid_5.json", "w") as f:
    json.dump(solution_details_global_without_self_correction_grid_5, f, indent=4)
with open("solutions/solution_details_global_without_self_correction_grid_7.json", "w") as f:
    json.dump(solution_details_global_without_self_correction_grid_7, f, indent=4)
with open("solutions/solution_details_global_without_self_correction_grid_10.json", "w") as f:
    json.dump(solution_details_global_without_self_correction_grid_10, f, indent=4)
with open("solutions/solution_details_local_self_correction_grid_5.json", "w") as f:
    json.dump(solution_details_local_self_correction_grid_5, f, indent=4)
with open("solutions/solution_details_local_self_correction_grid_7.json", "w") as f:
    json.dump(solution_details_local_self_correction_grid_7, f, indent=4)
with open("solutions/solution_details_local_self_correction_grid_10.json", "w") as f:
    json.dump(solution_details_local_self_correction_grid_10, f, indent=4)
with open("solutions/solution_details_local_without_self_correction_grid_5.json", "w") as f:
    json.dump(solution_details_local_without_self_correction_grid_5, f, indent=4)
with open("solutions/solution_details_local_without_self_correction_grid_7.json", "w") as f:
    json.dump(solution_details_local_without_self_correction_grid_7, f, indent=4)
with open("solutions/solution_details_local_without_self_correction_grid_10.json", "w") as f:
    json.dump(solution_details_local_without_self_correction_grid_10, f, indent=4)  